In [0]:

from pyspark.sql import functions as F

df = (
    spark.read.table("default.silver_gold_18K")
    .filter(F.col("date") >= F.date_sub(F.current_date(), 30))
)

In [0]:
pdf = df.toPandas()

In [0]:
pdf = pdf.rename(columns={"date": "ds", "price": "y"})

In [0]:
%pip install prophet

In [0]:
import pandas as pd
from prophet import Prophet
from datetime import datetime

# Último dia registrado
last_date = pdf['ds'].max()

# Último dia do mês corrente
end_month = pd.Timestamp(year=last_date.year, month=last_date.month, day=1) + pd.offsets.MonthEnd(1)

# Quantos dias faltam?
days_to_predict = (pd.Timestamp(end_month) - pd.Timestamp(last_date)).days
days_to_predict

In [0]:
# Treinar o modelo
model = Prophet()
model.fit(pdf)

future = model.make_future_dataframe(periods=days_to_predict)
forecast = model.predict(future)

# Apenas previsões para o restante do mês
last_date = pd.Timestamp(last_date)
forecast_restante = forecast[forecast['ds'] > last_date]


In [0]:
forecast_spark = spark.createDataFrame(forecast_restante[['ds', 'yhat', 'yhat_lower', 'yhat_upper']])

In [0]:
forecast_spark.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/invest/forecast_monthly_prediction")
forecast_spark.write.mode("overwrite").saveAsTable("forecast_monthly_prediction")